# #22C — Epitope Prediction/Design via AI-ML (ESM2 Fine-Tuning + Generation)

Training our own real model to predict MHC-I binding directly from peptide sequence -- distinct from #22,
which only called IEDB's own existing NetMHCpan tool via its API -- by fine-tuning a pretrained ESM2 protein
language model on real labeled IEDB binding data, then using the trained model generatively to propose novel
candidate peptides. Read PROJECT_NARRATIVE.md and WORKFLOW.md before starting.

## Step 0 — environment setup + fetch real IEDB binding data

Real source: IEDB's own compiled retraining dataset (`binding_data_2013.zip` -- compiled by IEDB together
with the Sette and Buus labs, the same real data family NetMHCpan/MHCflurry themselves train on). Verified
not reachable from the teacher's sandbox (network-restricted there); **the very first real thing you do with
it is print its real columns before trusting any assumed format** -- see Step 1.

In [1]:
!pip install -q transformers torch

import os
import zipfile
import io
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import pearsonr, spearmanr

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

IEDB_BINDING_DATA_URL = "https://tools.iedb.org/static/main/binding_data_2013.zip"

resp = requests.get(IEDB_BINDING_DATA_URL)
resp.raise_for_status()
with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
    names = zf.namelist()
    print("files in zip:", names)
    zf.extractall("iedb_binding_data")

print("setup complete")

files in zip: ['bdata.20130222.mhci.txt']
setup complete


## Step 1 — confirm the real column layout, clean, transform

**Biology/stats reason:** real binding-affinity data is never training-ready as downloaded -- real censored
measurements (an inequality like "&gt;" meaning the true IC50 is at least this high, not an exact value),
real non-human alleles, and real peptide lengths outside the 8-15mer MHC-I range all need handling before
this can be a clean supervised regression target.
**What·Why·How:** load the extracted real file with `pd.read_csv(..., sep="\t")` (print `.columns` FIRST and
confirm it matches the expected real layout before assuming anything); filter to real human (`HLA-`) alleles
and real peptide lengths 8-15; drop or appropriately handle real inequality-qualified rows (a simple, honest
first pass: keep only rows where the real measurement is an exact "=" value, not "&gt;"/"&lt;"); apply
`y = 1 - log(ic50) / log(50000)` clipped to [0, 1] (look up `np.clip`).

In [3]:
raw_path = "iedb_binding_data"
df = pd.read_csv("iedb_binding_data/bdata.20130222.mhci.txt", sep="\t")
print(df.columns.tolist())
print(df.head())

df_human = df[df['mhc'].str.startswith('HLA-')]
df_len = df_human[df_human['peptide_length'].between(8, 15)]
df_exact = df_len[df_len['inequality'] == '=']

def ic50_to_target(ic50):
    y = 1 - np.log(ic50) / np.log(50000)
    return np.clip(y, 0, 1)

df_exact = df_exact.copy()
df_exact["target"] = ic50_to_target(df_exact["meas"])
print(df_exact[["sequence", "mhc", "meas", "target"]].describe(include="all"))
print("real rows after cleaning:", len(df_exact))

['species', 'mhc', 'peptide_length', 'sequence', 'inequality', 'meas']
      species          mhc  peptide_length  sequence inequality          meas
0  chimpanzee  Patr-A*0101               8  RRDYRRGL          =    778.583409
1  chimpanzee  Patr-A*0101               8  YHSNVKEL          =  18806.166640
2  chimpanzee  Patr-A*0101               8  AQFSPQYL          =  22203.186860
3  chimpanzee  Patr-A*0101               8  GDYKLVEI          >  87128.712870
4  chimpanzee  Patr-A*0101               8  RGYVFQGL          >  87128.712870
         sequence          mhc          meas        target
count       86123        86123  8.612300e+04  86123.000000
unique      27931          118           NaN           NaN
top     YTAVVPLVY  HLA-A*02:01           NaN           NaN
freq           39         8681           NaN           NaN
mean          NaN          NaN  1.129339e+04      0.413287
std           NaN          NaN  7.832317e+04      0.281230
min           NaN          NaN  1.000000e+00    

## Step 2 — tokenize with ESM2's own tokenizer, allele-aware split

**Biology/stats reason:** splitting randomly by row risks the exact same real peptide/allele pair
appearing in both train and validation (many real rows are duplicate measurements of the same pair from
different real labs) -- a real train/test leakage that would make validation performance look better than
it honestly is.
**What·Why·How:** load `AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")`; tokenize real
`sequence` values; use `sklearn.model_selection.GroupShuffleSplit` grouped by the real combined
`sequence`+`mhc` key so the same real pair never splits across train/val.

In [4]:
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")

df_exact["pair_key"] = df_exact["sequence"] + "_" + df_exact["mhc"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(df_exact, groups=df_exact["pair_key"]))

train_df = df_exact.iloc[train_idx].reset_index(drop=True)
val_df = df_exact.iloc[val_idx].reset_index(drop=True)

print("real train rows:", len(train_df), "| real val rows:", len(val_df))
print("real overlap check (should be 0):",
      len(set(train_df["pair_key"]) & set(val_df["pair_key"])))

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

real train rows: 73204 | real val rows: 12919
real overlap check (should be 0): 0


## Step 3 — fine-tune ESM2 with a regression head

**Biology/stats reason:** ESM2's pretrained weights already encode general real protein sequence chemistry;
fine-tuning end-to-end (not freezing the backbone) lets that general knowledge adapt specifically to the
narrower real task of MHC-I binding affinity.
**What·Why·How:** build an `nn.Module` wrapping a pretrained `AutoModel.from_pretrained(...)` ESM2 backbone
plus one `nn.Linear` regression head on the real pooled (mean over real token embeddings, excluding padding)
sequence representation; train with `torch.optim.Adam` minimizing real `nn.functional.mse_loss` against the
real `target` column, batching real tokenized sequences.

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

class ESM2BindingRegressor(nn.Module):
    def __init__(self, backbone_name="facebook/esm2_t12_35M_UR50D"):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_name)
        hidden_size = self.backbone.config.hidden_size
        self.head = nn.Linear(hidden_size, 1) # nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        token_embeds = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (token_embeds * mask).sum(1) / mask.sum(1)
        return self.head(pooled).squeeze(-1)

model = ESM2BindingRegressor().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

batch_size = 32
n_epochs = 3

def batches(df, bs):
    for i in range(0, len(df), bs):
        yield df.iloc[i:i+bs]

for epoch in range(n_epochs):
    model.train()
    total_loss = 0.0
    for batch_df in batches(train_df.sample(frac=1, random_state=epoch), batch_size):
        enc = tokenizer(list(batch_df["sequence"]), return_tensors="pt", padding=True).to(device)
        targets = torch.tensor(batch_df["target"].values, dtype=torch.float32).to(device)

        optimizer.zero_grad()
        preds = model(enc["input_ids"], enc["attention_mask"])
        loss = nn.functional.mse_loss(preds, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_df)

    print(f"epoch {epoch}: real mean train MSE = {total_loss / len(train_df):.4f}")

using device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  136MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 0: real mean train MSE = 0.0672
epoch 1: real mean train MSE = 0.0563
epoch 2: real mean train MSE = 0.0501


## Step 4 — evaluate real validation performance

**Biology/stats reason:** Pearson/Spearman correlation between predicted and real measured affinity is the
standard real metric this literature reports (not raw MSE alone), letting an honest real comparison against
published NetMHCpan4.1/MHCflurry2.0 benchmark numbers.
**What·Why·How:** run the fine-tuned model over `val_df` in `eval()`/`no_grad()` mode, collect real
predictions, compute `scipy.stats.pearsonr` and `spearmanr` against the real `target` column.

In [6]:
model.eval()
val_preds = []
with torch.no_grad():
    for batch_df in batches(val_df, batch_size):
        enc = tokenizer(list(batch_df["sequence"]), return_tensors="pt", padding=True).to(device)
        preds = model(enc["input_ids"], enc["attention_mask"])
        val_preds.extend(preds.cpu().numpy().tolist())

val_df = val_df.copy()
val_df["predicted"] = val_preds

pearson_r, pearson_p = pearsonr(val_df["target"], val_df["predicted"])
spearman_r, spearman_p = spearmanr(val_df["target"], val_df["predicted"])

print(f"real validation Pearson r = {pearson_r:.4f} (p={pearson_p:.2e})")
print(f"real validation Spearman r = {spearman_r:.4f} (p={spearman_p:.2e})")

real validation Pearson r = 0.5505 (p=0.00e+00)
real validation Spearman r = 0.5500 (p=0.00e+00)


## Step 5 — generation: propose and score candidate mutants

**Biology/stats reason:** this is an honest propose-and-score loop, not a dedicated generative architecture
(see WORKFLOW.md) -- it answers "can the fine-tuned predictor itself be used to search for better real
candidates," a genuinely useful real application even though it's simpler than a true generative model.
**What·Why·How:** pick one real high-affinity peptide from `train_df` (highest real `target`); generate every
real single-residue substitution (20 real amino acids, excluding the original residue at each position);
score every real mutant with the fine-tuned model; keep the real top 10 novel candidates by predicted score.

In [7]:
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

seed_row = train_df.loc[train_df["target"].idxmax()]
seed_seq = seed_row["sequence"]
print("real seed peptide:", seed_seq, "| real training target:", seed_row["target"])

def generate_single_mutants(seq):
    mutants = []
    for i, orig in enumerate(seq):
        for aa in AMINO_ACIDS:
            if  aa != orig:
                mutants.append(seq[:i] + aa + seq[i+1:])
    return mutants

candidates = generate_single_mutants(seed_seq)
print("real candidate mutants generated:", len(candidates))

model.eval()
cand_scores = []
with torch.no_grad():
    for i in range(0, len(candidates), batch_size):
        batch = candidates[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True).to(device)
        scores = model(enc["input_ids"], enc["attention_mask"])
        cand_scores.extend(scores.cpu().numpy().tolist())

cand_df = pd.DataFrame({"sequence": candidates, "predicted_score": cand_scores})
cand_df = cand_df.sort_values("predicted_score", ascending=False)
top_candidates = cand_df.head(10)
print(top_candidates)

real seed peptide: CSDAFYPFY | real training target: 1.0
real candidate mutants generated: 171
      sequence  predicted_score
160  CSDAFYPFK         0.532250
47   CSMAFYPFY         0.525911
156  CSDAFYPFF         0.521984
6    ISDAFYPFY         0.502909
114  CSDAFYAFY         0.487761
91   CSDATYPFY         0.484117
166  CSDAFYPFR         0.482155
9    MSDAFYPFY         0.475238
129  CSDAFYTFY         0.470214
0    ASDAFYPFY         0.470115


## Step 6 — export results

In [9]:
val_df[["sequence", "mhc", "target", "predicted"]].to_csv(
    os.path.join(RESULTS_DIR, "validation_predictions.csv"), index=False)

with open(os.path.join(RESULTS_DIR, "validation_metrics.txt"), "w") as f:
    f.write(f"pearson_r={pearson_r}\npearson_p={pearson_p}\n")
    f.write(f"spearman_r={spearman_r}\nspearman_p={spearman_p}\n")
    f.write(f"n_train={len(train_df)}\nn_val={len(val_df)}\n")

top_candidates.to_csv(os.path.join(RESULTS_DIR, "generated_candidates.csv"), index=False)

print("exported:", os.listdir(RESULTS_DIR))

exported: ['validation_predictions.csv', 'generated_candidates.csv', 'validation_metrics.txt']
